# 02 — Preference Alignment with DPO
### Stage 3: turning the SFT model into "PostTraining Tutor - v2"

This notebook loads the SFT adapter from Notebook 1 and further trains it with **Direct Preference Optimization (DPO)** on `preference_dataset.jsonl`, so the model learns to prefer better explanations over worse ones for the same question.


## Step 0 — Install dependencies

**What/Why:** same reasoning as Notebook 1 - Unsloth for fast LoRA training, TRL for `DPOTrainer`.

In [ ]:
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U trl peft accelerate bitsandbytes transformers datasets


In [ ]:
# WHAT: imports for DPO training.
import torch
import json
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import DPOTrainer, DPOConfig


## Step 1 — Load the SFT model (base + SFT adapter)

**What:** reload the same base model and re-attach the LoRA adapter saved at the end of Notebook 1.
**Why:** DPO in this project is applied *on top of* the SFT model - we are refining an already instruction-tuned model's preferences, not starting from the raw base model.
**How:** load the base 4-bit model exactly as before, then load the saved adapter directory as the starting LoRA weights (instead of creating a fresh, randomly-initialized adapter).

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
SFT_ADAPTER_DIR = "outputs/sft_adapter"   # from Notebook 1 - change if you pushed to the Hub instead

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# WHAT: attach the SFT LoRA adapter (not a fresh one) so DPO starts from SFT behavior.
model.load_adapter(SFT_ADAPTER_DIR, adapter_name="default")
FastLanguageModel.for_training(model)   # WHY: switches Unsloth back into training mode.


## Step 2 — Prepare the preference dataset

**What:** load `preference_dataset.jsonl`, where each row has `prompt`, `chosen`, and `rejected` fields.
**Why:** DPO needs *paired* responses to the same prompt - one preferred, one not - so it can directly increase the model's relative probability of the chosen response versus the rejected one.
**How:** `DPOTrainer` expects a dataset with exactly these three columns (as raw text, not yet tokenized) - it handles tokenization and log-probability computation internally.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

pref_rows = load_jsonl("data/preference_dataset.jsonl")
print(f"Loaded {len(pref_rows)} preference pairs")
print(pref_rows[0])

# WHAT: wrap each prompt in the chat template (system + user turn), leaving
#       chosen/rejected as plain assistant-response text - DPOTrainer appends
#       them to the templated prompt internally when computing log-probs.
def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are PostTraining Tutor, an assistant that explains LLM training concepts clearly and concisely."},
        {"role": "user", "content": example["prompt"]},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {
        "prompt": prompt_text,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

dpo_dataset = Dataset.from_list(pref_rows).map(format_prompt)


## Step 3 — Configure DPOTrainer

**What DPO optimizes:** for each (prompt, chosen, rejected) triple, DPO increases the model's log-probability of `chosen` relative to `rejected`, relative to a frozen reference model - all in a single supervised-style loss, with **no separate reward model and no RL rollout loop** (unlike classic RLHF/PPO).

**Key parameters explained:**
- `beta=0.1`: controls how strongly the model is pushed away from the reference (SFT) model's behavior. Lower `beta` -> larger, more aggressive preference updates; higher `beta` -> more conservative, stays closer to the SFT model. `0.1`-`0.5` is a typical starting range.
- `learning_rate=5e-6`: DPO learning rates are usually much lower than SFT's - we are fine-tuning *preferences* on top of an already-good model, and a large LR here can quickly destabilize fluency.
- `num_train_epochs=1-2`: preference datasets are typically small; too many epochs risks overfitting to the specific chosen/rejected pairs rather than learning the general preference pattern.
- **Reference model**: because we're using PEFT/LoRA, `DPOTrainer` can use the *same* underlying base model with the adapter disabled as the implicit reference - no need to keep a second full copy of the model in memory (a big practical win for Colab).

In [ ]:
dpo_args = DPOConfig(
    output_dir="outputs/stage3_dpo",
    beta=0.1,                         # preference strength vs. staying close to the SFT model
    per_device_train_batch_size=1,    # DPO computes log-probs for 2 sequences per example -> more memory per step
    gradient_accumulation_steps=8,    # effective batch size = 1*8 = 8, matching Notebook 1's scale
    num_train_epochs=2,
    learning_rate=5e-6,               # deliberately much lower than the SFT stage's 2e-4
    logging_steps=5,
    optim="adamw_8bit",
    warmup_steps=5,
    lr_scheduler_type="linear",
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,
    seed=42,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,          # WHY None: with a PEFT model, TRL uses the base model
                              #           (adapter disabled) as the reference automatically.
    args=dpo_args,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

dpo_trainer.train()


## Step 4 — Save the DPO-aligned adapter

**What/Why:** same reasoning as Notebook 1 - save only the (now further-updated) LoRA adapter, this time under a separate directory so we can still load the SFT-only version independently in evaluation.

In [ ]:
DPO_ADAPTER_DIR = "outputs/dpo_adapter"
model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)
print(f"Saved DPO adapter to {DPO_ADAPTER_DIR}")

# Optional Hub upload - same pattern as Notebook 1:
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
# model.push_to_hub("your-username/postraining-tutor-dpo-adapter")


## Step 5 — Inference comparison: SFT vs. DPO

**What:** ask the same questions and compare the SFT-only adapter's answers to the DPO-aligned adapter's answers.
**Why:** this is the most direct, human-readable evidence of what DPO changed - look for differences in conciseness, confidence, or hedging.

In [ ]:
FastLanguageModel.for_inference(model)  # currently holds the DPO adapter

test_questions = [
    "What is LoRA and why is it useful for fine-tuning large language models?",
    "Explain the difference between SFT and DPO in simple terms.",
    "What does RLHF stand for and how does it relate to DPO?",
]

def generate(model, question):
    messages = [
        {"role": "system", "content": "You are PostTraining Tutor, an assistant that explains LLM training concepts clearly and concisely."},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    output = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== DPO model answers ===")
for q in test_questions:
    print(f"Q: {q}\nA: {generate(model, q)}\n{'-'*80}")

# NOTE: to see the SFT-only answers for a true side-by-side, either:
#   (a) disable the adapter temporarily: `model.disable_adapters()` then re-enable, or
#   (b) reload the SFT adapter from Notebook 1 in a fresh session.
# Notebook 3 does this properly by loading all three checkpoints at once.


**Next:** open `03_Evaluation.ipynb` to compare Base vs. SFT vs. DPO side by side on the fixed evaluation question set.